In [71]:
import os
import shutil
import re
from dotenv import load_dotenv
from typing import List, Tuple
from dotenv import load_dotenv
import tiktoken

from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain.schema import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [72]:
# Function get all sub-directory under root directory
def get_all_subdirectories_under_one_level(root_dir: str):
    """
    Get all subdirectories directly under the given root directory (one level only).
    Return a list of full paths.
    """
    return [
        os.path.join(root_dir, name)
        if not os.path.isabs(name) else name
        for name in os.listdir(root_dir)
        if os.path.isdir(os.path.join(root_dir, name))
    ]
def get_app_package_name(path: str) -> str:
    """
    Return the last directory name from a full path.
    Example:
        E:\\wearable-2-dataset-code-extraction\\adarshurs.android.vlcmobileremote
        -> 'adarshurs.android.vlcmobileremote'
    """
    return os.path.basename(os.path.normpath(path))
def create_directory_if_not_exist(path: str):
    """Tạo thư mục nếu chưa tồn tại."""
    if not os.path.exists(path):
        os.makedirs(path)
def copy_file(file_path, dest_dir):
    """Copy file vào thư mục đích."""
    shutil.copy(file_path, dest_dir)

In [73]:
# ----------------- helpers -----------------
# Day la ham co de quy, không can dung
# def _walk_files(root: str, suffix: Tuple[str, ...] = (".java",)) -> List[str]:
#     out = []
#     for dp, _, files in os.walk(root):
#         for fn in files:
#             if fn.lower().endswith(suffix):
#                 out.append(os.path.join(dp, fn))
#     return out
def _list_java_files_top_level(root: str) -> list[str]:
    """Chỉ lấy các file .java ngay dưới thư mục root (không đi vào subdirectory)."""
    try:
        names = os.listdir(root)
    except Exception:
        return []
    out = []
    for fn in names:
        fp = os.path.join(root, fn)
        if os.path.isfile(fp) and fn.lower().endswith(".java"):
            out.append(fp)
    return out
def _read_text(path: str) -> str:
    try:
        with open(path, "r", encoding="utf-8", errors="ignore") as f:
            return f.read()
    except Exception:
        return ""

def _batch_by_token_budget(texts: List[str], enc, max_tokens: int) -> List[List[str]]:
    """
    Gom nhóm 'texts' sao cho tổng tokens mỗi batch <= max_tokens.
    """
    batches, cur, cur_tokens = [], [], 0
    for t in texts:
        n = len(enc.encode(t))
        # nếu text đơn lẻ đã vượt budget -> vẫn nhét riêng 1 batch (API sẽ tự từ chối nếu quá dài)
        if n > max_tokens and not cur:
            batches.append([t])
            continue
        if cur_tokens + n > max_tokens and cur:
            batches.append(cur)
            cur, cur_tokens = [t], n
        else:
            cur.append(t)
            cur_tokens += n
    if cur:
        batches.append(cur)
    return batches

# ----------------- main -----------------
def find_similar_java_files_with_openai_embeddings_safe(
    code_files_root: str,
    code_blocks_root: str,
    embedding_model: str = "text-embedding-3-small",
    chunk_size: int = 1500,
    chunk_overlap: int = 200,
    top_k_per_block: int = 5,
    MAX_REQUEST_TOKENS: int = 280_000,  # < 300k để chừa headroom
) -> List[str]:
    """
    Trả về list đường dẫn .java trong `code_files_root` giống các code block ở `code_blocks_root`
    (OpenAI Embeddings + FAISS), có batching theo ngân sách token để tránh lỗi 400.
    """
    # 1) API key
    load_dotenv()
    api_key = os.getenv("OPENAI_API_KEY")
    if not api_key:
        raise ValueError("OPENAI_API_KEY is not set in the environment variables")

    # 2) Encoder cho đếm token (OpenAI embeddings dùng cl100k_base)
    enc = tiktoken.get_encoding("cl100k_base")

    # 3) Chuẩn bị code files -> chunk -> Documents
    file_paths = _list_java_files_top_level(code_files_root)
    if not file_paths:
        return []

    splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap)
    docs: List[Document] = []
    for fp in file_paths:
        txt = _read_text(fp)
        if not txt.strip():
            continue
        for ch in splitter.split_text(txt):
            docs.append(Document(page_content=ch, metadata={"path": fp}))
    if not docs:
        return []

    # 4) Chuẩn bị embeddings + FAISS, nhưng build theo batch token
    embeddings = OpenAIEmbeddings(model=embedding_model, api_key=api_key)

    texts = [d.page_content for d in docs]
    metas = [d.metadata for d in docs]
    # batch theo token
    text_batches = _batch_by_token_budget(texts, enc, MAX_REQUEST_TOKENS)
    meta_batches = []
    idx = 0
    for b in text_batches:
        meta_batches.append(metas[idx: idx + len(b)])
        idx += len(b)

    # Build FAISS gia tăng
    vs = None
    for tb, mb in zip(text_batches, meta_batches):
        if vs is None:
            vs = FAISS.from_texts(tb, embedding=embeddings, metadatas=mb)
        else:
            vs.add_texts(tb, metadatas=mb)

    # 5) Load code blocks (.java) và search theo batch để không vượt limit
    block_paths = _walk_files(code_blocks_root, suffix=(".java",))
    matched_files = set()

    # Có thể block ngắn, nhưng vẫn batch an toàn
    block_texts = []
    block_map_idx = []  # lưu index -> path
    for bp in block_paths:
        bt = _read_text(bp).strip()
        if bt:
            block_texts.append(bt)
            block_map_idx.append(bp)

    if not block_texts:
        return []

    block_batches = _batch_by_token_budget(block_texts, enc, MAX_REQUEST_TOKENS)

    # Với mỗi block, truy vấn top-k và gom file path
    start = 0
    for bb in block_batches:
        for j, q in enumerate(bb):
            results = vs.similarity_search(q, k=max(1, top_k_per_block))
            for r in results:
                matched_files.add(r.metadata.get("path"))
        start += len(bb)

    return sorted(matched_files)

In [74]:
code_file_root_directory = r"E:\wearable-2-sharing-compliance-code-extraction"
code_block_root_directory = r"C:\Users\ASUS\anaconda3-project-code\wearable-2-sharing-non-compliance-evaluation\java_blocks"

In [75]:
level_1_code_file_arr = get_all_subdirectories_under_one_level(code_file_root_directory)
# print(level_1_directory_arr)

In [76]:
# Load .env và khởi tạo model
load_dotenv()
api_key = os.getenv("OPENAI_API_KEY")
if not api_key:
    raise ValueError("OPENAI_API_KEY is not set in the environment variables")

llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0,
    openai_api_key=api_key
)

In [77]:
for i in range(len(level_1_directory_arr)):
    level_1_code_file_path = level_1_code_file_arr[i]
    similar_code_file_directoty = level_1_code_file_path+"\\code-file-similarity"
    create_directory_if_not_exist(similar_code_file_directoty)
    package_name = get_app_package_name(level_1_code_file_path)
    level_1_code_block_path = code_block_root_directory+"\\"+package_name
    print("----------------------------------- Loop-"+str(i)+"-----------------------------------")
    print("Package name: ",package_name)
    # print("level_1_code_file_path: ",level_1_code_file_path)
    # print("level_1_code_block_path: ",level_1_code_block_path)
    result = find_similar_java_files_with_openai_embeddings_safe(
        code_files_root=level_1_code_file_path,
        code_blocks_root=level_1_code_block_path,
        embedding_model="text-embedding-3-small",  # hoặc "text-embedding-3-large"
        chunk_size=1500,
        chunk_overlap=200,
        top_k_per_block=5,
        MAX_REQUEST_TOKENS=280_000  
    )
    print(f"✅ Found {len(result)} similar files:")
    for similar_file_path in result:
        copy_file(similar_file_path, similar_code_file_directoty)
    break

----------------------------------- Loop-0-----------------------------------
Package name:  app.groupcal.www.apk
✅ Found 40 similar files:
